# LeetCode 2149 — Rearrange Array Elements by Sign bằng Python

Link đề: https://leetcode.com/problems/rearrange-array-elements-by-sign/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Simulation

Skeleton LeetCode:

```python
class Solution:
    def rearrangeArray(self, nums: List[int]) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Đọc kỹ ba điều kiện và nhận ra **đáp án là duy nhất**: dương ở chỉ số chẵn, âm ở chỉ số lẻ, mỗi nhóm giữ nguyên thứ tự xuất hiện (**ổn định** — stable).
2. Cài đặt các cách: vét cạn hoán vị (kiểm chứng), "tìm phần tử kế tiếp" O(n²), **tách hai danh sách rồi xen kẽ**, **hai con trỏ ghi vào vị trí chẵn/lẻ**.
3. Hiểu khái niệm **phân hoạch ổn định** (stable partition) và vì sao làm **tại chỗ** (in-place) mà vẫn ổn định là khó hơn nhiều.
4. Thành thạo `zip`, list comprehension lồng, gán theo lát cắt `ans[0::2] = pos`.
5. Viết hàm kiểm tra **tính hợp lệ** của một đáp án (thay vì chỉ so khớp) để test chắc chắn hơn.

## 2. Đề bài & ví dụ

Cho mảng `nums` độ dài **chẵn**, gồm **số lượng bằng nhau** số dương và số âm (không có số 0). Trả về mảng sắp xếp lại sao cho:
1. Mọi cặp **liền kề** trái dấu nhau.
2. Với các số **cùng dấu**, thứ tự xuất hiện ban đầu được **giữ nguyên**.
3. Mảng **bắt đầu bằng số dương**.

Không bắt buộc làm tại chỗ.

**Ví dụ 1:** `nums = [3,1,-2,-5,2,-4]` → `[3,-2,1,-5,2,-4]`. Dương theo thứ tự: 3, 1, 2; âm: −2, −5, −4.

**Ví dụ 2:** `nums = [-1,1]` → `[1,-1]`.

**Ràng buộc:** `2 ≤ nums.length ≤ 2·10⁵`, độ dài chẵn, `1 ≤ |nums[i]| ≤ 10⁵`, số dương = số âm.

In [1]:
from typing import List
from itertools import permutations
import random, time

examples = [([3, 1, -2, -5, 2, -4], [3, -2, 1, -5, 2, -4]), ([-1, 1], [1, -1])]
def is_valid(nums, out):
    # kiểm tra out có thỏa 3 điều kiện so với nums không
    if sorted(out) != sorted(nums): return False
    if any((a > 0) == (b > 0) for a, b in zip(out, out[1:])): return False
    if out and out[0] < 0: return False
    pos = [x for x in nums if x > 0]; neg = [x for x in nums if x < 0]
    return [x for x in out if x > 0] == pos and [x for x in out if x < 0] == neg

for nums, exp in examples:
    print(nums, "->", exp, "hợp lệ:", is_valid(nums, exp))
print("[1,-2,2,-5,3,-4] hợp lệ?", is_valid(examples[0][0], [1, -2, 2, -5, 3, -4]))

[3, 1, -2, -5, 2, -4] -> [3, -2, 1, -5, 2, -4] hợp lệ: True
[-1, 1] -> [1, -1] hợp lệ: True
[1,-2,2,-5,3,-4] hợp lệ? False


## 3. Kiến thức nền

### 3.1 Lọc theo điều kiện với list comprehension (giữ thứ tự)

`[x for x in nums if x > 0]` duyệt từ trái sang phải nên **giữ nguyên thứ tự** — đây chính là tính ổn định cần thiết.

In [2]:
nums = [3, 1, -2, -5, 2, -4]
pos = [x for x in nums if x > 0]; neg = [x for x in nums if x < 0]
print("dương:", pos, "| âm:", neg)

dương: [3, 1, 2] | âm: [-2, -5, -4]


### 3.2 `zip` và "làm phẳng" danh sách cặp

`zip(pos, neg)` cho các cặp `(p, n)`; làm phẳng bằng comprehension hai vòng `for`.

In [3]:
pairs = list(zip(pos, neg))
print(pairs)
print([x for pair in pairs for x in pair])

[(3, -2), (1, -5), (2, -4)]
[3, -2, 1, -5, 2, -4]


### 3.3 Gán theo lát cắt có bước nhảy

`ans[0::2]` là các vị trí chẵn `0, 2, 4, …`; `ans[1::2]` là vị trí lẻ. Gán một list đúng độ dài vào lát cắt có bước nhảy sẽ điền đúng các vị trí đó.

In [4]:
ans = [None] * 6
ans[0::2] = pos
ans[1::2] = neg
print(ans)
try:
    ans[0::2] = [1, 2]               # sai độ dài
except ValueError as e:
    print("ValueError:", e)

[3, -2, 1, -5, 2, -4]
ValueError: attempt to assign sequence of size 2 to extended slice of size 3


### 3.4 Sắp xếp ổn định (`sorted` của Python là stable)

`sorted(nums, key=lambda x: x < 0)` đưa mọi số dương (khóa `False`) lên trước số âm (khóa `True`), giữ thứ tự trong mỗi nhóm — vì Timsort **ổn định**.

In [5]:
print(sorted(nums, key=lambda x: x < 0))

[3, 1, 2, -2, -5, -4]


## 4. Ý tưởng nền tảng / trực giác

Từ điều kiện 1 + 3: mảng kết quả có dạng `+, −, +, −, …` ⇒ **vị trí chẵn là dương, vị trí lẻ là âm**.

Từ điều kiện 2: số dương thứ `t` (theo thứ tự xuất hiện) phải nằm ở vị trí dương thứ `t`, tức chỉ số `2t`; số âm thứ `t` ở chỉ số `2t + 1`.

⇒ Đáp án **duy nhất**: `ans[2t] = pos[t]`, `ans[2t + 1] = neg[t]`.

Tính tay ví dụ 1:

| t | pos[t] → ans[2t] | neg[t] → ans[2t+1] |
|---|---|---|
| 0 | 3 → ans[0] | −2 → ans[1] |
| 1 | 1 → ans[2] | −5 → ans[3] |
| 2 | 2 → ans[4] | −4 → ans[5] |

`ans = [3, −2, 1, −5, 2, −4]`. ✔

In [6]:
for t, (p, q) in enumerate(zip(pos, neg)):
    print(f"t={t}: ans[{2*t}] = {p}, ans[{2*t+1}] = {q}")

t=0: ans[0] = 3, ans[1] = -2
t=1: ans[2] = 1, ans[3] = -5
t=2: ans[4] = 2, ans[5] = -4


**Tại sao không có lựa chọn nào?** Ba điều kiện "khóa chặt" lẫn nhau: điều kiện 1 + 3 cố định *loại* (dấu) của từng vị trí; điều kiện 2 cố định *phần tử cụ thể* trong mỗi loại. Nhiều bài "sắp xếp lại" khác cho phép nhiều đáp án (ví dụ LC 905 — chỉ cần chẵn trước lẻ sau), khi đó phải viết hàm kiểm tra tính hợp lệ thay vì so khớp. Ở đây ta có cả hai: `is_valid` để kiểm tra và so khớp trực tiếp vì đáp án duy nhất.

**Đếm nhanh:** nếu bỏ điều kiện 2 (không cần giữ thứ tự) thì số đáp án là `(n/2)!·(n/2)!` — với `n = 6` là 36. Điều kiện ổn định thu hẹp về đúng 1.

In [7]:
from math import factorial
def count_without_order(nums):
    # đếm số mảng xen kẽ dấu, bắt đầu dương, là hoán vị của nums (bỏ điều kiện thứ tự) - các phần tử coi là phân biệt
    h = len(nums) // 2
    return factorial(h) ** 2
print("n=6, bỏ điều kiện thứ tự:", count_without_order([3, 1, -2, -5, 2, -4]), "cách; giữ thứ tự: 1 cách")

n=6, bỏ điều kiện thứ tự: 36 cách; giữ thứ tự: 1 cách


## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn hoán vị (chỉ để kiểm chứng tính duy nhất)

In [8]:
def brute_perm(nums):
    sols = {p for p in permutations(nums) if is_valid(nums, list(p))}
    return [list(p) for p in sols]

for nums, exp in examples:
    sols = brute_perm(nums)
    print(nums, "-> số đáp án hợp lệ:", len(sols), sols)

[3, 1, -2, -5, 2, -4] -> số đáp án hợp lệ: 1 [[3, -2, 1, -5, 2, -4]]
[-1, 1] -> số đáp án hợp lệ: 1 [[1, -1]]


Đúng một đáp án — khớp lập luận ở mục 4. Độ phức tạp O(n!·n), chỉ dùng n ≤ 8.

### 5.2 Baseline O(n²): mỗi vị trí đi tìm phần tử phù hợp chưa dùng đầu tiên

Ở vị trí chẵn tìm **số dương chưa dùng đầu tiên** (quét từ đầu), ở vị trí lẻ tìm **số âm chưa dùng đầu tiên**.

In [9]:
def brute_scan(nums):
    n = len(nums); used = [False] * n; ans = []
    for k in range(n):
        want_pos = (k % 2 == 0)
        for i in range(n):                         # quét lại từ đầu mỗi lần -> O(n^2)
            if not used[i] and (nums[i] > 0) == want_pos:
                used[i] = True; ans.append(nums[i]); break
    return ans

for nums, exp in examples:
    print(brute_scan(nums), "OK" if brute_scan(nums) == exp else "SAI")

[3, -2, 1, -5, 2, -4] OK
[1, -1] OK


**Độ phức tạp:** O(n²) — với `n = 2·10⁵` là 4·10¹⁰ ⇒ TLE.

**Nút thắt:** mỗi lần ta quét lại **từ đầu** dù biết chắc mọi phần tử trước "lần tìm thấy gần nhất" đã dùng hoặc sai dấu. Chỉ cần nhớ **vị trí tìm thấy gần nhất** cho mỗi dấu ⇒ hai con trỏ.

In [10]:
for n in [2000, 6000]:
    a = [random.randint(1, 9) for _ in range(n // 2)] + [-random.randint(1, 9) for _ in range(n // 2)]
    t = time.perf_counter(); brute_scan(a); print(f"n={n}: {(time.perf_counter()-t)*1000:.0f} ms")

n=2000: 40 ms


n=6000: 314 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** đáp án được xác định hoàn toàn: dương thứ `t` → chỉ số `2t`, âm thứ `t` → chỉ số `2t + 1`. Do đó chỉ cần **một lượt** duyệt `nums` với hai bộ đếm:
- `i` = vị trí chẵn kế tiếp cho số dương (bắt đầu 0, mỗi lần +2);
- `j` = vị trí lẻ kế tiếp cho số âm (bắt đầu 1, mỗi lần +2).

**Bất biến:** sau khi duyệt tiền tố `nums[0..s]`, nếu tiền tố có `a` số dương và `b` số âm thì `i = 2a`, `j = 2b + 1`, và `ans[0], ans[2], …, ans[2a−2]` là `a` số dương đầu theo đúng thứ tự (tương tự cho âm).

**Quy tắc:** **nếu** `x > 0` **thì** `ans[i] = x; i += 2`; **ngược lại** `ans[j] = x; j += 2`.

**Hai cách nhìn tương đương:**
- "Phân phối" (distribute): mỗi phần tử biết ngay mình đi đâu.
- "Hợp nhất" (merge): hai con trỏ trên input (`p` tìm dương kế tiếp, `q` tìm âm kế tiếp), lấy luân phiên. Mỗi con trỏ chỉ đi tới, nên tổng O(n).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tách hai danh sách rồi xen kẽ

In [11]:
def split_interleave(nums):
    pos = [x for x in nums if x > 0]
    neg = [x for x in nums if x < 0]
    return [x for pair in zip(pos, neg) for x in pair]

for nums, exp in examples:
    print(split_interleave(nums), "OK" if split_interleave(nums) == exp else "SAI")

[3, -2, 1, -5, 2, -4] OK
[1, -1] OK


Biến thể dùng gán lát cắt (rất "Pythonic" và nhanh):

In [12]:
def slice_assign(nums):
    ans = [0] * len(nums)
    ans[0::2] = [x for x in nums if x > 0]
    ans[1::2] = [x for x in nums if x < 0]
    return ans

for nums, exp in examples:
    print(slice_assign(nums), "OK" if slice_assign(nums) == exp else "SAI")

[3, -2, 1, -5, 2, -4] OK
[1, -1] OK


**Độ phức tạp cách 2:** O(n) thời gian, O(n) bộ nhớ phụ (hai danh sách tạm, mỗi cái n/2) ngoài mảng kết quả.

### 7.2 Cách 3 — Một lượt, ghi vào vị trí chẵn/lẻ (cách doocs)

In [13]:
def two_slots(nums):
    ans = [0] * len(nums)
    i, j = 0, 1
    for x in nums:
        if x > 0:
            ans[i] = x; i += 2
        else:
            ans[j] = x; j += 2
    return ans

for nums, exp in examples:
    print(two_slots(nums), "OK" if two_slots(nums) == exp else "SAI")

[3, -2, 1, -5, 2, -4] OK
[1, -1] OK


**Độ phức tạp cách 3:** O(n) thời gian, O(1) bộ nhớ phụ ngoài mảng kết quả.

### 7.3 Cách 4 — Hai con trỏ trên input (hợp nhất luân phiên)

`p` luôn trỏ tới số dương **chưa dùng** đầu tiên, `q` tới số âm chưa dùng đầu tiên. Mỗi bước lấy `nums[p]` rồi `nums[q]`, sau đó đẩy mỗi con trỏ tới phần tử cùng dấu kế tiếp.

In [14]:
def merge_pointers(nums):
    n = len(nums); ans = []
    p = q = 0
    for _ in range(n // 2):
        while nums[p] < 0: p += 1
        while nums[q] > 0: q += 1
        ans.append(nums[p]); ans.append(nums[q])
        p += 1; q += 1
    return ans

for nums, exp in examples:
    print(merge_pointers(nums), "OK" if merge_pointers(nums) == exp else "SAI")

[3, -2, 1, -5, 2, -4] OK
[1, -1] OK


Vì sao không tràn chỉ số? Ở vòng thứ `t` (tính từ 0) đã dùng `t` số dương, và còn `n/2 − t ≥ 1` số dương chưa dùng ⇒ vòng `while nums[p] < 0` chắc chắn dừng trước khi `p` vượt `n − 1`. Mỗi con trỏ chỉ tăng, tổng số bước ≤ `n` mỗi con trỏ ⇒ O(n).

### 7.4 Cách 5 — Sắp xếp ổn định theo khóa "vị trí đích"

Gán cho mỗi phần tử khóa = chỉ số đích (`2t` hoặc `2t + 1`) rồi sắp xếp. O(n log n) — kém hơn nhưng minh họa ý "đáp án là một hoán vị xác định".

In [15]:
def sort_by_target(nums):
    cp = cn = 0; keys = []
    for x in nums:
        if x > 0: keys.append(2 * cp); cp += 1
        else: keys.append(2 * cn + 1); cn += 1
    return [x for _, x in sorted(zip(keys, nums))]

for nums, exp in examples:
    print(sort_by_target(nums), "OK" if sort_by_target(nums) == exp else "SAI")

[3, -2, 1, -5, 2, -4] OK
[1, -1] OK


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1.

| x | dấu | ghi vào | i sau | j sau | ans |
|---|---|---|---|---|---|
| 3 | + | ans[0] | 2 | 1 | [3,_,_,_,_,_] |
| 1 | + | ans[2] | 4 | 1 | [3,_,1,_,_,_] |
| −2 | − | ans[1] | 4 | 3 | [3,−2,1,_,_,_] |
| −5 | − | ans[3] | 4 | 5 | [3,−2,1,−5,_,_] |
| 2 | + | ans[4] | 6 | 5 | [3,−2,1,−5,2,_] |
| −4 | − | ans[5] | 6 | 7 | [3,−2,1,−5,2,−4] |

In [16]:
def trace_two_slots(nums):
    ans = ["_"] * len(nums); i, j = 0, 1
    for x in nums:
        if x > 0:
            ans[i] = x; where = i; i += 2
        else:
            ans[j] = x; where = j; j += 2
        print(f"x={x:>3}: ghi vào ans[{where}] -> i={i}, j={j}, ans={ans}")
    return ans

trace_two_slots([3, 1, -2, -5, 2, -4])

x=  3: ghi vào ans[0] -> i=2, j=1, ans=[3, '_', '_', '_', '_', '_']
x=  1: ghi vào ans[2] -> i=4, j=1, ans=[3, '_', 1, '_', '_', '_']
x= -2: ghi vào ans[1] -> i=4, j=3, ans=[3, -2, 1, '_', '_', '_']
x= -5: ghi vào ans[3] -> i=4, j=5, ans=[3, -2, 1, -5, '_', '_']
x=  2: ghi vào ans[4] -> i=6, j=5, ans=[3, -2, 1, -5, 2, '_']
x= -4: ghi vào ans[5] -> i=6, j=7, ans=[3, -2, 1, -5, 2, -4]


[3, -2, 1, -5, 2, -4]

Trace cách 4 (hai con trỏ trên input) trên một mảng mà các số âm dồn ở đầu:

In [17]:
def trace_merge(nums):
    n = len(nums); ans = []; p = q = 0
    for t in range(n // 2):
        while nums[p] < 0: p += 1
        while nums[q] > 0: q += 1
        ans += [nums[p], nums[q]]
        print(f"vòng {t}: p={p} (nums[p]={nums[p]}), q={q} (nums[q]={nums[q]}) -> ans={ans}")
        p += 1; q += 1
    return ans

trace_merge([-1, -2, -3, 4, 5, 6])

vòng 0: p=3 (nums[p]=4), q=0 (nums[q]=-1) -> ans=[4, -1]
vòng 1: p=4 (nums[p]=5), q=1 (nums[q]=-2) -> ans=[4, -1, 5, -2]
vòng 2: p=5 (nums[p]=6), q=2 (nums[q]=-3) -> ans=[4, -1, 5, -2, 6, -3]


[4, -1, 5, -2, 6, -3]

## 9. Code nộp LeetCode

In [18]:
from typing import List

class Solution:
    def rearrangeArray(self, nums: List[int]) -> List[int]:
        ans = [0] * len(nums)
        i, j = 0, 1                  # vị trí chẵn cho số dương, lẻ cho số âm
        for x in nums:
            if x > 0:
                ans[i] = x
                i += 2
            else:
                ans[j] = x
                j += 2
        return ans

In [19]:
sol = Solution()
for nums, exp in examples:
    got = sol.rearrangeArray(nums)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

[3, -2, 1, -5, 2, -4] | kỳ vọng [3, -2, 1, -5, 2, -4] | OK
[1, -1] | kỳ vọng [1, -1] | OK


## 10. Vì sao thuật toán đúng

**Tính duy nhất.** Gọi `out` là một đáp án hợp lệ. Điều kiện 1 (xen kẽ dấu) + điều kiện 3 (bắt đầu dương) ⇒ `out[k] > 0` ⇔ `k` chẵn. Có đúng `n/2` vị trí chẵn và `n/2` số dương. Điều kiện 2 (giữ thứ tự) ⇒ dãy các số dương của `out` (đọc theo vị trí chẵn tăng dần) là `pos[0], pos[1], …` ⇒ `out[2t] = pos[t]`. Tương tự `out[2t+1] = neg[t]`. Vậy `out` duy nhất.

**Thuật toán tạo đúng dãy đó.** Theo bất biến ở mục 6, số dương thứ `t` gặp trong lượt duyệt được ghi vào `ans[2t]`, số âm thứ `t` vào `ans[2t + 1]`. Vì số dương = số âm = `n/2`, cuối cùng `i = n` và `j = n + 1` — không ghi ra ngoài mảng, và mọi ô đều được ghi đúng một lần. ∎

Mục 12 kiểm chứng tính duy nhất bằng vét cạn hoán vị trên input nhỏ.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space phụ (ngoài kết quả) |
|---|---|---|
| 1a. Vét cạn hoán vị | O(n!·n) | O(n) |
| 1b. Tìm kế tiếp, quét lại | O(n²) | O(n) |
| 2. Tách + xen kẽ / gán lát cắt | O(n) | O(n) |
| 3. Một lượt chẵn/lẻ | O(n) | O(1) |
| 4. Hai con trỏ trên input | O(n) | O(1) |
| 5. Sắp xếp theo khóa đích | O(n log n) | O(n) |

In [20]:
rng = random.Random(2149)
N = 2 * 10**5
big = [rng.randint(1, 10**5) for _ in range(N // 2)] + [-rng.randint(1, 10**5) for _ in range(N // 2)]
rng.shuffle(big)
res = {}
for name, f in [("split_interleave", split_interleave), ("slice_assign", slice_assign), ("two_slots", two_slots),
                ("merge_pointers", merge_pointers), ("sort_by_target", sort_by_target), ("Solution", Solution().rearrangeArray)]:
    t = time.perf_counter(); res[name] = f(big); dt = time.perf_counter() - t
    print(f"{name:<17} {dt*1000:6.1f} ms")
assert all(v == res["Solution"] for v in res.values()) and is_valid(big, res["Solution"])

split_interleave    25.8 ms


slice_assign        17.7 ms
two_slots            7.3 ms


merge_pointers      22.6 ms
sort_by_target      73.0 ms
Solution             7.1 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [21]:
ALL = {"brute_scan": brute_scan, "split_interleave": split_interleave, "slice_assign": slice_assign,
       "two_slots": two_slots, "merge_pointers": merge_pointers, "sort_by_target": sort_by_target,
       "Solution": Solution().rearrangeArray}
edge = examples + [
    ([1, -1], [1, -1]),
    ([-3, -2, -1, 1, 2, 3], [1, -3, 2, -2, 3, -1]),     # âm dồn đầu
    ([1, 2, 3, -1, -2, -3], [1, -1, 2, -2, 3, -3]),     # dương dồn đầu
    ([5, -5, 5, -5], [5, -5, 5, -5]),                   # đã đúng dạng
    ([2, 2, -2, -2], [2, -2, 2, -2]),                   # giá trị trùng
]
for nums, exp in edge:
    for name, f in ALL.items():
        a = list(nums)
        assert f(a) == exp, (name, nums, f(list(nums)), exp)
        assert a == nums, (name, "đã sửa input")
print("Edge case OK:", len(edge))

Edge case OK: 7


### 12.2 Kiểm chứng tính duy nhất bằng vét cạn hoán vị (n ≤ 6)

In [22]:
rng = random.Random(42)
for _ in range(300):
    h = rng.randint(1, 3)                     # n <= 6 để vét cạn hoán vị nhanh
    nums = [rng.randint(1, 5) for _ in range(h)] + [-rng.randint(1, 5) for _ in range(h)]
    rng.shuffle(nums)
    sols = brute_perm(nums)
    assert len(sols) == 1
    for name, f in ALL.items():
        assert f(list(nums)) == sols[0], (name, nums)
print("300 case: đáp án duy nhất và mọi cách đều khớp")

300 case: đáp án duy nhất và mọi cách đều khớp


### 12.3 Ngẫu nhiên lớn hơn: kiểm tra bằng `is_valid`

In [23]:
rng = random.Random(7)
for _ in range(500):
    h = rng.randint(1, 300)
    nums = [rng.randint(1, 10**5) for _ in range(h)] + [-rng.randint(1, 10**5) for _ in range(h)]
    rng.shuffle(nums)
    out = Solution().rearrangeArray(nums)
    assert is_valid(nums, out)
    for name in ["split_interleave", "slice_assign", "two_slots", "merge_pointers", "sort_by_target"]:
        assert ALL[name](nums) == out
print("500 case vừa: hợp lệ")

500 case vừa: hợp lệ


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp xếp theo giá trị (phá thứ tự xuất hiện)

In [24]:
def wrong_sort_values(nums):
    pos = sorted(x for x in nums if x > 0); neg = sorted((x for x in nums if x < 0), reverse=True)
    return [x for pair in zip(pos, neg) for x in pair]

print("bản sai :", wrong_sort_values([3, 1, -2, -5, 2, -4]))
print("bản đúng:", Solution().rearrangeArray([3, 1, -2, -5, 2, -4]))

bản sai : [1, -2, 2, -4, 3, -5]
bản đúng: [3, -2, 1, -5, 2, -4]


**Vì sao sai:** điều kiện 2 yêu cầu giữ **thứ tự xuất hiện**, không phải thứ tự giá trị. **Sửa:** lọc theo dấu và giữ nguyên thứ tự.

### Lỗi 2 — Bắt đầu bằng số âm (đảo vai chẵn/lẻ)

In [25]:
def wrong_start_negative(nums):
    ans = [0] * len(nums); i, j = 1, 0             # SAI: dương vào lẻ, âm vào chẵn
    for x in nums:
        if x > 0: ans[i] = x; i += 2
        else: ans[j] = x; j += 2
    return ans

print("bản sai :", wrong_start_negative([3, 1, -2, -5, 2, -4]))
print("bản đúng:", Solution().rearrangeArray([3, 1, -2, -5, 2, -4]))

bản sai : [-2, 3, -5, 1, -4, 2]
bản đúng: [3, -2, 1, -5, 2, -4]


**Sửa:** `i = 0` (dương ở chẵn), `j = 1` (âm ở lẻ).

### Lỗi 3 — Hoán đổi tham lam tại chỗ (phá tính ổn định)

In [26]:
def wrong_inplace_swap(nums):
    a = list(nums); n = len(a)
    for k in range(n):
        want_pos = (k % 2 == 0)
        if (a[k] > 0) != want_pos:
            for t in range(k + 1, n):
                if (a[t] > 0) == want_pos:
                    a[k], a[t] = a[t], a[k]          # SAI: hoán đổi làm xáo thứ tự nhóm kia
                    break
    return a

for nums in ([3, 1, -2, -5, 2, -4], [1, 2, 3, -1, -2, -3]):
    out = wrong_inplace_swap(nums)
    print(f"{nums}: bản sai {out} (hợp lệ? {is_valid(nums, out)}) | bản đúng {Solution().rearrangeArray(nums)}")

[3, 1, -2, -5, 2, -4]: bản sai [3, -2, 1, -5, 2, -4] (hợp lệ? True) | bản đúng [3, -2, 1, -5, 2, -4]
[1, 2, 3, -1, -2, -3]: bản sai [1, -1, 3, -2, 2, -3] (hợp lệ? False) | bản đúng [1, -1, 2, -2, 3, -3]


Ví dụ 1 của đề tình cờ vẫn ra đúng. Với `[1, 2, 3, −1, −2, −3]`: ở `k = 1` hoán đổi `2` ↔ `−1` ⇒ `[1, −1, 3, 2, −2, −3]`; ở `k = 3` hoán đổi `2` ↔ `−2` ⇒ `[1, −1, 3, −2, 2, −3]` — số dương thành `1, 3, 2`, **sai thứ tự**.

**Vì sao sai:** hoán đổi đưa một phần tử đang "đứng chờ" (ở đây là `2`) nhảy ra **phía sau** các phần tử cùng nhóm với nó. Muốn tại chỗ mà vẫn ổn định phải **xoay** (rotate) đoạn `a[k..t]` (dịch cả khối) — O(n²) (xem biến thể 15.1). **Sửa:** dùng mảng kết quả mới.

### Lỗi 4 — Dùng `x >= 0` vs `x > 0` — không sao ở bài này, nhưng `x > 0` vs `x < 0` lẫn lộn ở hai nơi thì sai

In [27]:
def wrong_mixed_conditions(nums):
    pos = [x for x in nums if x > 0]
    neg = [x for x in nums if x > 0]                 # SAI: chép nhầm điều kiện
    return [x for pair in zip(pos, neg) for x in pair]

print("bản sai :", wrong_mixed_conditions([3, 1, -2, -5, 2, -4]))
print("bản đúng:", Solution().rearrangeArray([3, 1, -2, -5, 2, -4]))

bản sai : [3, 3, 1, 1, 2, 2]
bản đúng: [3, -2, 1, -5, 2, -4]


**Sửa:** viết một hàm phân loại duy nhất (`x > 0`) và dùng `else` cho nhóm còn lại, như ở cách 3 — tránh chép–sửa hai điều kiện. (Đề đảm bảo không có số 0 nên `x > 0` / `else` là đủ.)

### Lỗi 5 — `zip` cắt cụt khi dùng cho biến thể số lượng không bằng nhau

In [28]:
def interleave_unequal_wrong(nums):
    pos = [x for x in nums if x > 0]; neg = [x for x in nums if x < 0]
    return [x for pair in zip(pos, neg) for x in pair]        # SAI nếu len(pos) != len(neg)

nums = [1, 2, 3, -1]                                           # biến thể: dư số dương, nối cuối
print("bản sai (mất phần tử):", interleave_unequal_wrong(nums))
from itertools import zip_longest
print("bản đúng:", [x for pair in zip_longest([1, 2, 3], [-1]) for x in pair if x is not None])

bản sai (mất phần tử): [1, -1]
bản đúng: [1, -1, 2, 3]


Ở bài gốc số lượng luôn bằng nhau nên `zip` an toàn; nhưng khi tái dùng code cho LC 2149 biến thể (LC 2161 / "dư thì nối cuối") phải dùng `zip_longest` hoặc nối phần dư.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Giữ nguyên thứ tự trong mỗi nhóm" ⇒ **phân hoạch ổn định**: lọc ra các danh sách theo nhóm (giữ thứ tự), hoặc ghi thẳng vào vị trí đích.
- Khi vị trí đích của mỗi phần tử **tính được ngay** ⇒ một lượt "phân phối" O(n).
- "Không bắt buộc tại chỗ" ⇒ đừng tự làm khó; tại chỗ + ổn định thường O(n²) hoặc cần thuật toán phức tạp.
- Python: `ans[0::2] = …` là cách gọn nhất để điền xen kẽ.
- Bài cùng họ: 905 Sort Array By Parity (không cần ổn định), 922 Sort Array By Parity II, 2161 Partition Array According to Given Pivot (ổn định), 75 Sort Colors.

## 15. Biến thể của bài

### 15.1 Biến thể A — Làm **tại chỗ** và **ổn định** (O(n²), O(1) bộ nhớ)

Ở mỗi vị trí `k` sai dấu, tìm phần tử đúng dấu gần nhất `t > k` rồi **xoay** đoạn `a[k..t]` sang phải 1 (đưa `a[t]` về `k`, dịch cả khối) — giữ thứ tự cả hai nhóm.

In [29]:
def inplace_stable(a):
    n = len(a)
    for k in range(n):
        want_pos = (k % 2 == 0)
        if (a[k] > 0) != want_pos:
            t = k + 1
            while (a[t] > 0) != want_pos: t += 1
            x = a[t]
            for s in range(t, k, -1):          # dịch khối a[k..t-1] sang phải
                a[s] = a[s - 1]
            a[k] = x
    return a

for nums, exp in examples + [([-3, -2, -1, 1, 2, 3], None)]:
    print(nums, "->", inplace_stable(list(nums)))
rng = random.Random(151)
for _ in range(300):
    h = rng.randint(1, 30)
    nums = [rng.randint(1, 9) for _ in range(h)] + [-rng.randint(1, 9) for _ in range(h)]; rng.shuffle(nums)
    assert inplace_stable(list(nums)) == Solution().rearrangeArray(nums)
print("tại chỗ ổn định: khớp")

[3, 1, -2, -5, 2, -4] -> [3, -2, 1, -5, 2, -4]
[-1, 1] -> [1, -1]
[-3, -2, -1, 1, 2, 3] -> [1, -3, 2, -2, 3, -1]
tại chỗ ổn định: khớp


### 15.2 Biến thể B — Số lượng **không bằng nhau**: xen kẽ hết mức có thể rồi nối phần dư vào cuối (giữ thứ tự)

In [30]:
def rearrange_unequal(nums):
    pos = [x for x in nums if x > 0]; neg = [x for x in nums if x < 0]
    k = min(len(pos), len(neg))
    out = [x for pair in zip(pos[:k], neg[:k]) for x in pair]
    return out + pos[k:] + neg[k:]

print(rearrange_unequal([1, 2, 3, -1]), rearrange_unequal([-1, -2, 3, -4, -5]), rearrange_unequal([28, -41, 22, -8, -37, 46, 35, -9, 18, -6, 19, -26, -37, -10, -9, 15, 14, 31]))

[1, -1, 2, 3] [3, -1, -2, -4, -5] [28, -41, 22, -8, 46, -37, 35, -9, 18, -6, 19, -26, 15, -37, 14, -10, 31, -9]


### 15.3 Biến thể C — Bắt đầu bằng số âm, hoặc nhóm theo **chẵn/lẻ** thay vì dấu

Chỉ đổi hàm phân loại `key(x)` và vị trí bắt đầu. Viết tổng quát:

In [31]:
def interleave_by(nums, key, first=True):
    ans = [0] * len(nums)
    i, j = (0, 1) if first else (1, 0)
    for x in nums:
        if key(x): ans[i] = x; i += 2
        else: ans[j] = x; j += 2
    return ans

print(interleave_by([3, 1, -2, -5, 2, -4], lambda x: x > 0, first=False))   # bắt đầu bằng âm
print(interleave_by([4, 2, 5, 7, 6, 9], lambda x: x % 2 == 0))               # chẵn ở vị trí chẵn (LC 922 bản ổn định)

[-2, 3, -5, 1, -4, 2]
[4, 5, 2, 7, 6, 9]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1a. Hoán vị | thử hết | O(n!·n) | O(n) | kiểm chứng duy nhất |
| 1b. Tìm kế tiếp | quét lại từ đầu | O(n²) | O(n) | minh họa nút thắt |
| 2. Tách + xen kẽ | hai list rồi zip / gán lát cắt | O(n) | O(n) | code ngắn, Pythonic |
| 3. Ghi chẵn/lẻ | một lượt phân phối | O(n) | O(1) | **mặc định** |
| 4. Hai con trỏ input | hợp nhất luân phiên | O(n) | O(1) | khi muốn sinh dần (streaming) |
| 5. Sort theo khóa đích | hoán vị xác định | O(n log n) | O(n) | minh họa |
| 15.1 Tại chỗ ổn định | xoay khối | O(n²) | O(1) | bị cấm cấp phát mảng mới |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `rearrangeArray([-5, -4, 3, 2])` và `rearrangeArray([7, -1, -2, 8, 9, -3])`?

**Bài 2 (tính tay):** Trong `nums = [4, -7, -1, 6, -3, 5]`, số `-1` sẽ nằm ở chỉ số nào trong kết quả? Số `5`?

**Bài 3 (điền TODO):** Viết `bt3_todo(nums)` trả về **mảng chỉ số đích** `dest` với `dest[i]` = vị trí của `nums[i]` trong kết quả.

**Bài 4 (tự viết lại):** Viết `bt4_todo(nums)` dùng cách 4 (hai con trỏ trên input) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(nums)` **đảo ngược** phép biến đổi: cho kết quả `out` hợp lệ và **chuỗi dấu** của mảng gốc (`signs`, list `'+'`/`'-'`), dựng lại `nums`. (Hàm nhận `(out, signs)`.)

In [32]:
def bt3_todo(nums):
    # TODO: dest[i] = vị trí của nums[i] trong kết quả
    pass

def bt4_todo(nums):
    # TODO: hai con trỏ trên input
    pass

def bt5_todo(out, signs):
    # TODO: dựng lại nums từ kết quả và chuỗi dấu gốc
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: dương theo thứ tự xuất hiện. Mức 2: `[3, 2]` và `[−5, −4]`; `[7, 8, 9]` và `[−1, −2, −3]`. Mức 3: `[3, −5, 2, −4]` và `[7, −1, 8, −2, 9, −3]`.

**Bài 2.** Mức 1: `−1` là số âm thứ mấy? Mức 2: thứ 2 (`t = 1`) ⇒ chỉ số `2t + 1`. Mức 3: `−1` ở chỉ số 3; `5` là số dương thứ 3 (`t = 2`) ⇒ chỉ số 4.

**Bài 3.** Mức 1: như cách 3 nhưng ghi chỉ số. Mức 2: `dest[i] = i_pos` hoặc `j_neg`. Mức 3: tăng con trỏ tương ứng thêm 2.

**Bài 4.** Mức 1: `p`, `q` bắt đầu ở 0. Mức 2: `while nums[p] < 0: p += 1`. Mức 3: lấy `nums[p]`, `nums[q]` rồi tăng cả hai.

**Bài 5.** Mức 1: `out[0::2]` là các số dương theo thứ tự gốc. Mức 2: duyệt `signs`, gặp `'+'` lấy số dương kế tiếp. Mức 3: hai con trỏ trên `out[0::2]` và `out[1::2]`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `[3,−5,2,−4]`, `[7,−1,8,−2,9,−3]`. **Bài 2:** `−1` ở chỉ số 3, `5` ở chỉ số 4.

In [33]:
assert Solution().rearrangeArray([-5, -4, 3, 2]) == [3, -5, 2, -4]
assert Solution().rearrangeArray([7, -1, -2, 8, 9, -3]) == [7, -1, 8, -2, 9, -3]
out = Solution().rearrangeArray([4, -7, -1, 6, -3, 5])
assert out.index(-1) == 3 and out.index(5) == 4

def bt3(nums):
    dest = []; i, j = 0, 1
    for x in nums:
        if x > 0: dest.append(i); i += 2
        else: dest.append(j); j += 2
    return dest

def bt4(nums):
    n = len(nums); ans = []; p = q = 0
    for _ in range(n // 2):
        while nums[p] < 0: p += 1
        while nums[q] > 0: q += 1
        ans += [nums[p], nums[q]]; p += 1; q += 1
    return ans

def bt5(out, signs):
    pos, neg = iter(out[0::2]), iter(out[1::2])
    return [next(pos) if s == "+" else next(neg) for s in signs]

rng = random.Random(1949)
for _ in range(1000):
    h = rng.randint(1, 20)
    nums = [rng.randint(1, 9) for _ in range(h)] + [-rng.randint(1, 9) for _ in range(h)]; rng.shuffle(nums)
    out = Solution().rearrangeArray(nums)
    d = bt3(nums)
    assert sorted(d) == list(range(len(nums))) and all(out[d[i]] == nums[i] for i in range(len(nums)))
    assert bt4(nums) == out
    assert bt5(out, ["+" if x > 0 else "-" for x in nums]) == nums
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


Ghi chú bài 5: phép biến đổi **không** khả nghịch nếu chỉ biết `out` (mất thông tin vị trí xen kẽ ban đầu) — cần thêm chuỗi dấu. Với chuỗi dấu, mỗi số dương/âm được "trả về" đúng thứ tự nhờ tính ổn định.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ans = [0] * len(nums)
i, j = 0, 1                  # dương -> chẵn, âm -> lẻ
for x in nums:
    if x > 0: ans[i] = x; i += 2
    else:     ans[j] = x; j += 2
return ans
# hoặc: ans[0::2] = [x for x in nums if x > 0]; ans[1::2] = [x for x in nums if x < 0]
```

**Checklist:**
- [ ] Bắt đầu bằng **dương** (`i = 0`).
- [ ] Giữ **thứ tự xuất hiện** trong mỗi nhóm (không sắp xếp theo giá trị, không hoán đổi tùy tiện).
- [ ] Một điều kiện phân loại duy nhất + `else`.
- [ ] Không sửa `nums` nếu không cần.
- [ ] Đã thử: âm dồn đầu, dương dồn đầu, giá trị trùng.